In [1]:
import os

print(os.path.abspath(os.curdir))

d:\Data Kuliah\Semester 9\Data in Brief\dataset\notebooks


In [2]:
import os
os.chdir("..")

In [3]:
print(os.path.abspath(os.curdir))

d:\Data Kuliah\Semester 9\Data in Brief\dataset


In [14]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn3, venn3_circles
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
import itertools

def extract_ngrams(texts, n=4, min_freq=2):
    """
    Extract n-grams from texts with minimum frequency threshold.
    
    Parameters:
    - texts: list of text strings
    - n: n-gram size (e.g., 4 for 4-grams)
    - min_freq: minimum frequency for an n-gram to be included
    """
    vectorizer = CountVectorizer(
        ngram_range=(n, n),  # Single n value
        min_df=min_freq,
        lowercase=True,
        token_pattern=r'\b\w+\b'
    )
    
    try:
        X = vectorizer.fit_transform(texts)
        ngrams = vectorizer.get_feature_names_out()
        frequencies = np.asarray(X.sum(axis=0)).flatten()
        return set(ngrams), dict(zip(ngrams, frequencies))
    except:
        return set(), {}

def get_representative_phrases(phrases, freq_dict, max_phrases=3):
    """
    Get representative phrases sorted by frequency.
    
    Parameters:
    - phrases: set of phrases
    - freq_dict: dictionary mapping phrases to frequencies
    - max_phrases: maximum number of phrases to return
    """
    if not phrases:
        return []
    
    # Sort by frequency (sum across all classes that have this phrase)
    sorted_phrases = sorted(phrases, key=lambda x: sum([freq_dict.get(label, {}).get(x, 0) 
                                                         for label in freq_dict.keys()]), 
                           reverse=True)
    return sorted_phrases[:max_phrases]

def create_single_venn_diagram(sets, labels, class_ngram_freq, output_path, 
                               max_phrases=3, n=4, min_freq=2, figsize=(10, 8), dpi=300):
    """
    Create a single 3-way Venn diagram with phrases.
    
    Parameters:
    - sets: list of 3 sets (one per class)
    - labels: list of 3 class labels
    - class_ngram_freq: dictionary of n-gram frequencies per class
    - output_path: path to save the figure
    - max_phrases: max phrases to show per region
    - n: n-gram size
    - min_freq: minimum frequency used
    - figsize: figure size
    - dpi: resolution
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = ['#ff9999', '#66b3ff', '#99ff99']
    
    # Create 3-way Venn diagram
    v = venn3(sets, set_labels=labels, ax=ax, alpha=0.5)
    venn3_circles(sets, ax=ax, linewidth=2.5)
    
    # Customize colors for individual sets
    if v:
        patch_ids = ['100', '010', '001']
        for i, patch_id in enumerate(patch_ids):
            patch = v.get_patch_by_id(patch_id)
            if patch:
                patch.set_color(colors[i])
                patch.set_alpha(0.5)
    
    # Calculate intersections and add phrases as text
    set_dict = {
        '100': sets[0] - sets[1] - sets[2],
        '010': sets[1] - sets[0] - sets[2],
        '001': sets[2] - sets[0] - sets[1],
        '110': (sets[0] & sets[1]) - sets[2],
        '101': (sets[0] & sets[2]) - sets[1],
        '011': (sets[1] & sets[2]) - sets[0],
        '111': sets[0] & sets[1] & sets[2]
    }
    
    # Add representative phrases to each region
    for region_id, phrases in set_dict.items():
        if phrases and v.get_label_by_id(region_id):
            # Get representative phrases
            repr_phrases = get_representative_phrases(
                phrases, class_ngram_freq, max_phrases
            )
            
            # Format text
            if repr_phrases:
                text = '\n'.join([f'"{p}"' for p in repr_phrases])
                
                # Get the label object and update it
                label_obj = v.get_label_by_id(region_id)
                
                # Show count and phrases
                count_text = f"n={len(phrases)}\n{text}"
                label_obj.set_text(count_text)
                label_obj.set_fontsize(9)
                label_obj.set_fontstyle('italic')
    
    # plt.title(f'Overlapping {n}-gram Phrases: {" ∩ ".join(labels)}\n' + 
    #           f'(min frequency: {min_freq}, showing top {max_phrases} phrases per region)', 
    #           fontsize=12, fontweight='bold', pad=15)
    
    plt.tight_layout()
    
    # Save figure
    plt.savefig(output_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close()
    
    print(f"✓ Saved: {output_path}")

def create_venn_diagrams_separated(data_path, output_dir='venn_diagrams', 
                                   n=4, min_freq=2, top_n=50, 
                                   max_phrases_per_region=3, figsize=(10, 8), dpi=300):
    """
    Create 4 separate Venn diagrams as independent files.
    
    Parameters:
    - data_path: path to CSV file with 'message' and 'label' columns
    - output_dir: directory to save output images (will create if doesn't exist)
    - n: n-gram size (e.g., 4 for 4-grams)
    - min_freq: minimum frequency for n-gram inclusion
    - top_n: number of top n-grams to consider per class
    - max_phrases_per_region: max phrases to show in each Venn region
    - figsize: figure size for each diagram
    - dpi: image resolution
    """
    import os
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Load data
    train_df = pd.read_csv(os.path.join(data_path, 'filtered_train.csv'))
    test_df = pd.read_csv(os.path.join(data_path, 'filtered_test.csv'))
    df = pd.concat([train_df, test_df], ignore_index=True)
    
    # Get unique labels
    labels = sorted(df['label'].unique())
    if len(labels) != 4:
        print(f"Warning: Expected 4 classes, found {len(labels)}: {labels}")
    
    print(f"\n{'='*60}")
    print(f"Extracting {n}-grams from dataset...")
    print(f"{'='*60}")
    
    # Extract n-grams for each class
    class_ngrams = {}
    class_ngram_freq = {}
    
    for label in labels:
        texts = df[df['label'] == label]['message'].tolist()
        ngrams, freq = extract_ngrams(texts, n=n, min_freq=min_freq)
        
        # Keep only top_n most frequent n-grams per class
        if freq:
            top_ngrams = sorted(freq.items(), key=lambda x: x[1], reverse=True)[:top_n]
            class_ngrams[label] = set([ng for ng, _ in top_ngrams])
            class_ngram_freq[label] = dict(top_ngrams)
        else:
            class_ngrams[label] = set()
            class_ngram_freq[label] = {}
        
        print(f"  {label}: {len(class_ngrams[label])} {n}-grams")
    
    label_list = list(labels)
    
    # Define 4 different 3-way combinations
    combinations = [
        (0, 1, 2),  
        (0, 1, 3),  
        (0, 2, 3),  
        (1, 2, 3),  
    ]
    
    print(f"\n{'='*60}")
    print(f"Generating {len(combinations)} Venn diagrams...")
    print(f"{'='*60}\n")
    
    # Create each Venn diagram as a separate file
    for idx, combo in enumerate(combinations, 1):
        selected_labels = [label_list[i] for i in combo]
        sets = [class_ngrams[label] for label in selected_labels]
        
        # Create filename
        filename = f"venn_diagram_{idx}_{'_'.join(selected_labels)}.png"
        output_path = os.path.join(output_dir, filename)
        
        # Create the diagram
        create_single_venn_diagram(
            sets=sets,
            labels=selected_labels,
            class_ngram_freq=class_ngram_freq,
            output_path=output_path,
            max_phrases=max_phrases_per_region,
            n=n,
            min_freq=min_freq,
            figsize=figsize,
            dpi=dpi
        )
    
    # Print detailed overlap statistics
    print(f"\n{'='*60}")
    print("Representative Overlapping Phrases")
    print(f"{'='*60}")
    
    for r in range(2, 5):
        for combo in itertools.combinations(label_list, r):
            intersection = set.intersection(*[class_ngrams[label] for label in combo])
            if intersection:
                repr_phrases = get_representative_phrases(intersection, class_ngram_freq, 5)
                print(f"\n{' ∩ '.join(combo)} ({len(intersection)} total overlapping {n}-grams):")
                for phrase in repr_phrases:
                    freqs = [class_ngram_freq.get(label, {}).get(phrase, 0) for label in combo]
                    print(f"  - \"{phrase}\" (frequencies: {freqs})")
    
    print(f"\n{'='*60}")
    print(f"✓ All diagrams saved to: {output_dir}/")
    print(f"{'='*60}\n")
    
    return class_ngrams, class_ngram_freq

In [15]:

# Usage example
# if __name__ == "__main__":
# Adjust these parameters as needed
data_path = os.path.join('severity', 'unique')  # Change to your CSV file path

# Parameters
n = 3                      # Use 4-grams (4-word phrases)
min_freq = 1               # Minimum frequency for n-gram to be included
top_n = 50                 # Top N n-grams per class to consider
max_phrases_per_region = 3 # Show top 3 phrases in each Venn region
output_dir = os.path.join('venn_diagrams', str(n))     # Directory to save all diagrams

class_ngrams, class_freq = create_venn_diagrams_separated(
    data_path=data_path,
    output_dir=output_dir,
    n=n,
    min_freq=min_freq,
    top_n=top_n,
    max_phrases_per_region=max_phrases_per_region,
    figsize=(10, 8),
    dpi=300
)


Extracting 3-grams from dataset...
  High: 50 3-grams
  Low: 50 3-grams
  Medium: 50 3-grams
  Normal: 50 3-grams

Generating 4 Venn diagrams...

✓ Saved: venn_diagrams\3\venn_diagram_1_High_Low_Medium.png
✓ Saved: venn_diagrams\3\venn_diagram_2_High_Low_Normal.png
✓ Saved: venn_diagrams\3\venn_diagram_3_High_Medium_Normal.png
✓ Saved: venn_diagrams\3\venn_diagram_4_Low_Medium_Normal.png

Representative Overlapping Phrases

High ∩ Low (4 total overlapping 3-grams):
  - "contact dji support" (frequencies: [np.int64(4), np.int64(15)])
  - "dji support for" (frequencies: [np.int64(3), np.int64(5)])
  - "support for assistance" (frequencies: [np.int64(3), np.int64(4)])
  - "to take off" (frequencies: [np.int64(2), np.int64(4)])

High ∩ Medium (6 total overlapping 3-grams):
  - "return to home" (frequencies: [np.int64(2), np.int64(14)])
  - "contact dji support" (frequencies: [np.int64(4), np.int64(3)])
  - "dji support for" (frequencies: [np.int64(3), np.int64(3)])
  - "support for assist